# 02 · Ground truth: download and build the census tables

**Goal:** one clean row per locality from INEGI's 2020 census, with three definitions of poverty, Indigenous
population shares, and the Relative Wealth Index tile attached.
**Time:** about 15 minutes, mostly downloading.
**Produces:** `data/processed/*.parquet` (stays on your computer) and `results/real/tables/build_summary.json` (saved to GitHub).

In [ ]:
# Make the project importable; works in JupyterLab, VS Code and Colab without reinstalling.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from mapaudit import notebook as nb
from mapaudit import pipeline, report
from mapaudit.config import get_paths

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
MODE = nb.notebook_mode()  # "real" for you; CI sets MAPAUDIT_NOTEBOOK_MODE=demo to test this notebook
print("Mode:", MODE)

## 1. Download the census file and the wealth map

In [ ]:
if MODE == "real":
    from mapaudit.config import load_config
    from mapaudit.download import download_all

    for message in download_all(load_config("real"), get_paths("real").raw):
        print(message)
else:
    print("Demo mode: the build step generates synthetic files instead.")

If a download fails, download it by hand and put it in `data/raw/` under the name the message gives:

- **Census (ITER):** INEGI → Censo de Población y Vivienda 2020 → Datos abiertos → *Principales resultados
  por localidad (ITER)* → national CSV.
- **Relative Wealth Index:** the "Relative Wealth Index" dataset on the Humanitarian Data Exchange. If there
  is no Mexico file, see `docs/data_card.md` for the fallback and log it in the plan's deviations table.

## 2. Build the locality table

In [ ]:
summary = pipeline.build(MODE)
pd.Series({k: v for k, v in summary.items() if k != "label_loadings"}, name="value", dtype=object).to_frame()

## 3. Sanity checks: every row should say `pass`

In [ ]:
nb.sanity_checks(summary, MODE)

## 4. Look at the data

In [ ]:
loc = pd.read_parquet(get_paths(MODE).localities)
pd.Series({
    "localities": len(loc),
    "rural, in scope": int(loc["in_scope"].sum()),
    "with dwelling data": int(loc["has_dwelling_data"].sum()),
    "without dwelling data (mostly < 3 dwellings)": int((~loc["has_dwelling_data"]).sum()),
    "people living in those": int(loc.loc[~loc["has_dwelling_data"], "POBTOT"].sum()),
    "Indigenous localities (>= 40%)": int((loc["indigenous"] == 1).sum()),
}, name="count").to_frame()

In [ ]:
by_state = (
    loc.assign(ind_pop=loc["POBTOT"] * loc["ind_share"])
    .groupby("NOM_ENT")[["POBTOT", "ind_pop"]]
    .sum()
)
by_state["ind_share"] = by_state["ind_pop"] / by_state["POBTOT"]
by_state.sort_values("ind_share", ascending=False).head(10).round(3)

How each poverty definition is built (principal-component loadings; all should be positive):

In [ ]:
pd.DataFrame(summary["label_loadings"]).round(3)

## 5. Spot-check places you know

Compare a few rows with INEGI's online tables. **Clear this cell's output before saving to GitHub**
(right-click → Clear Outputs): the data card promises that no locality-level rows go to GitHub.

In [ ]:
PLACES = ["Chamula", "Tlaxiaco", "Valladolid"]  # replace with places you know
cols = ["cvegeo", "NOM_ENT", "NOM_MUN", "NOM_LOC", "POBTOT", "ind_hh_share", "no_elec", "no_water", "dirt_floor"]
loc.loc[loc["NOM_LOC"].str.contains("|".join(PLACES), case=False, na=False), cols].head(15)

## Your notes (for lab-notebook post 2)

- How many localities and people have suppressed data? Who lives there?
- Anything surprising in the Indigenous shares by state?
- Did the spot checks match INEGI's tables?

## Save your work to GitHub

1. Run the next cell (real data only). It runs the checks GitHub will run: formatting, lint, tests and the
   website build. Every row should say `pass`.
2. Commit and push what changed, with a message that says what you did. Nothing from `data/` goes to
   GitHub, and no notebook output may show locality-level rows.
3. On GitHub, the **Actions** tab reruns the checks; every run should go green within about 10 minutes.

In [ ]:
if MODE == "real":
    display(nb.run_checks())